<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

## 02-1 · Part 5: Linear, Nonlinear, Stochastic, and Robust Optimization

**Two formulations can use the same decision story but require different methods because their functions or uncertain inputs are represented differently.**

Parts 2 through 4 changed the constraints, decision domain, and objective structure one at a time. This part classifies the remaining features: function structure, response evaluation, and treatment of uncertainty.

| Part that changes | Classification |
|:---|:---|
| Mathematical form of $f,g,h$ | Linear or nonlinear |
| How $y$ is calculated | Direct algebraic or simulation-based |
| How external inputs are represented | Deterministic, stochastic, or robust |


### 1 · Function structure determines linear or nonlinear optimization

A continuous **linear optimization problem** has a linear objective and linear constraints:

> $\displaystyle \underset{x}{\operatorname{minimize}}\quad c^{\mathsf{T}}x\qquad\text{subject to}\qquad Ax\le b,\quad Cx=d.$

If the objective or any constraint is nonlinear in the optimization variables, the formulation is a **nonlinear optimization problem**. Squared terms, products between variables, \(\max\), and nonlinear simulations are common sources of nonlinearity.

The classroom transition \(F\) is affine for fixed external inputs, but the complete optimization formulation is nonlinear because discomfort uses squared deviations and energy uses \(u_t^2\).

The function structure and decision domain combine:

| Function structure | Decision domain | Common name |
|:---|:---|:---|
| Linear | Continuous | Linear program (LP) |
| Linear | Contains integer variables | Mixed-integer linear program (MILP) |
| Nonlinear | Continuous | Nonlinear program (NLP) |
| Nonlinear | Contains integer variables | Mixed-integer nonlinear program (MINLP) |

These names describe the formulation, not the solver. Several algorithms may target the same problem class.


### 2 · Response evaluation determines algebraic or simulation-based optimization

In a direct algebraic formulation, the objective and constraints are evaluated directly from \(x\). For example, \(f(x)=c^{\mathsf{T}}x\) and \(g(x)=Ax-b\) require no time-stepping system model.

In a simulation-based formulation, a simulator first produces responses:

> $\displaystyle y=\operatorname{Sim}(x),\qquad f=f(y),\qquad g=g(x,y).$

The classroom problem is simulation-based because \(\operatorname{Sim}\) repeats the physical state transition \(F\) for 12 steps and applies \(G\) to obtain \(D(u)\) and \(E(u)\). The symbol \(F\) remains the physical state transition; it is not reused as a vector objective.

Simulation-based does not automatically mean stochastic. A simulator with one fixed input path is deterministic.


### 3 · Input representation determines deterministic, stochastic, or robust optimization

Let \(\xi\) represent uncertain external inputs such as outdoor temperature or occupancy. The physical decision \(x\) is still chosen by the controller.

| Treatment of $\xi$ | Standard idea | Interpretation |
|:---|:---|:---|
| Deterministic | Fix one value or one input path | Optimize for the stated conditions |
| Stochastic | Assign probabilities $p_s$ to scenarios $\xi_s$ | Optimize an expectation or another risk measure |
| Robust | Allow any $\xi$ in an uncertainty set $\mathcal U$ | Protect against the worst allowed case |

A simple stochastic objective is

> $\displaystyle \underset{x}{\operatorname{minimize}}\quad \sum_{s=1}^{S}p_s f\!\left(\operatorname{Sim}(x;\xi_s)\right),\qquad \sum_{s=1}^{S}p_s=1.$

A corresponding robust objective is

> $\displaystyle \underset{x}{\operatorname{minimize}}\quad \max_{\xi\in\mathcal U} f\!\left(\operatorname{Sim}(x;\xi)\right).$

The classroom demonstrations in earlier lectures are deterministic because \(T_t^{\mathrm{out}}=31\,^\circ\mathrm C\) and \(N_t=20\) are fixed for all candidates. The next calculation holds the cooling decision fixed and changes only the outdoor-temperature scenario.


In [ ]:
import numpy as np

TIME_STEPS = 12
INITIAL_TEMPERATURE = 27.0
WEATHER_EXCHANGE = 0.12
OCCUPANT_HEAT = 0.012
COOLING_EFFECT = 0.45
OCCUPANTS = np.full(TIME_STEPS, 20.0)


def expand_decision(x):
    early_cooling, late_cooling = np.asarray(x, dtype=float)
    return np.r_[np.full(6, early_cooling), np.full(6, late_cooling)]


def evaluate_outdoor_scenario(x, outdoor_temperature):
    cooling = expand_decision(x)
    temperatures = [INITIAL_TEMPERATURE]
    for people, action in zip(OCCUPANTS, cooling):
        current = temperatures[-1]
        temperatures.append(
            current
            + WEATHER_EXCHANGE * (outdoor_temperature - current)
            + OCCUPANT_HEAT * people
            - COOLING_EFFECT * action
        )
    temperatures = np.asarray(temperatures)
    discomfort = np.sum(
        np.maximum(temperatures[1:] - 24.0, 0.0) ** 2
        + np.maximum(22.0 - temperatures[1:], 0.0) ** 2
    )
    energy = 0.5 * np.sum(cooling**2)
    return temperatures, float(discomfort), float(energy)


decision = (3.0, 2.0)
print(f"{'outdoor (°C)':>13} | {'final T (°C)':>12} | {'D':>8} | {'E':>6}")
print('-' * 50)
for outdoor in (29.0, 31.0, 33.0):
    temperatures, discomfort, energy = evaluate_outdoor_scenario(decision, outdoor)
    print(f"{outdoor:>13.1f} | {temperatures[-1]:>12.2f} | "
          f"{discomfort:>8.2f} | {energy:>6.2f}")


The fixed decision uses the same energy in every scenario, but the temperature path and discomfort change with the external input. Evaluating several scenarios does not by itself define a stochastic or robust problem. The formulation must also state how those scenario results are combined or constrained.


### 4 · One formulation receives several labels at the same time

The classification axes are independent. A problem is not only “continuous” or only “nonlinear.” Its complete description combines all relevant labels.

| Example formulation | Constraint structure | Domain | Objectives | Functions | Evaluation | Uncertainty |
|:---|:---|:---|:---|:---|:---|:---|
| Classroom cooling with $J$ | Generally constrained | Continuous | Single | Nonlinear | Simulation-based | Deterministic |
| Classroom cooling with $[D,E]^{\mathsf{T}}$ | Generally constrained | Continuous | Multiple | Nonlinear | Simulation-based | Deterministic |
| Component sizing with integer counts | Generally constrained | Mixed | Single | Possibly nonlinear | Direct or simulated | Deterministic or uncertain |
| Ship routing with discrete visits | Generally constrained | Discrete | Single or multiple | Usually nonlinear | Direct or simulated | Deterministic or uncertain |

For example, the original classroom problem is a **nonlinear, generally constrained, continuous, simulation-based, deterministic, single-objective optimization problem**. Changing \(J\) to \([D,E]^{\mathsf{T}}\) changes only the objective classification to multi-objective.


### 5 · Problem class and algorithm are different descriptions

A formulation states what is chosen, evaluated, and required. An algorithm states how candidate decisions are searched. Grid search, gradient-based methods, evolutionary algorithms, and particle swarm optimization can target different formulations when their assumptions are appropriate.

Changing the algorithm while keeping \(x,\mathcal X,\operatorname{Sim},f,g,h\) fixed leaves the optimization problem unchanged. Changing any of those formulation parts creates a different problem, even if the same algorithm is used.


### Takeaway

Read a formulation along independent classification axes:

> **constraints → unconstrained, bound, or general · domain → continuous, discrete, or mixed · objectives → single or multiple · functions → linear or nonlinear · evaluation → algebraic or simulated · uncertainty → deterministic, stochastic, or robust**

Changing one part may change one label while every other label stays fixed. This controlled comparison makes the problem class precise before an optimization algorithm is selected.
